# 06. Ames Housing: доменная информация, очистка данных и feature engineering

Датасет: **Ames Housing** (Dean De Cock, 2011) — продажи жилой
недвижимости в г. Эймс, штат Айова, 2006–2010 гг.

* Файл данных: `train.csv` — [Kaggle: Ames Iowa Housing Data
  (marcopale/housing)](https://www.kaggle.com/datasets/marcopale/housing?select=train.csv),
* Оригинальная статья: De Cock, D. (2011). *"Ames, Iowa: Alternative to the
  Boston Housing Data as an End of Semester Regression Project."*
  Journal of Statistics Education, 19(3).


Ноутбук закрывает первую часть плана:
1. Доменная информация — описания колонок, ограничения, гипотезы.
2. Подготовка данных — очистка выбросов и заполнение пропусков **с опорой
   на документацию источника**, а не наугад.
3. Feature engineering — признаки, осмысленные именно в контексте
   оценки недвижимости.

Результат сохраняется в `data/ames_prepared.csv` и используется в ноутбуке
`07_modeling_cross_validation.ipynb`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import modules.style as style

from os.path import join

sns.set_style("whitegrid")
%matplotlib inline

In [ ]:
RAW_PATH = join('data', 'train.csv')
df_raw = pd.read_csv(RAW_PATH)
print("Размер данных:", df_raw.shape)
df_raw.head(3)

## 1. Доменная информация

### 1.1 Откуда и что это за данные

Данные собраны Ames City Assessor's Office (офисом налогового асессора
города Эймс) для целей расчёта налогооблагаемой стоимости жилой
недвижимости — но, по словам автора, содержат ровно ту информацию,
которую типичный покупатель хотел бы знать перед покупкой дома. Каждая
строка — одна продажа одного объекта недвижимости за 2006–2010 гг.

### 1.2 Типы переменных (по официальной документации)

Согласно `DataDocumentation.txt`, 82 столбца делятся на:
* **23 номинальные** (категории без порядка: `Neighborhood`, `Bldg Type`,
  `Sale Type` и т.д.);
* **23 порядковые** (категории с естественным порядком качества/состояния:
  `Exter Qual`, `Kitchen Qual`, `Bsmt Qual` и т.д. — почти всегда по шкале
  `Ex > Gd > TA > Fa > Po`, иногда `NA` = "нет данной характеристики");
* **14 дискретных** (целочисленные счётчики: `Bedroom AbvGr`,
  `Fireplaces`, `Garage Cars` и т.д.);
* **20 непрерывных** (площади, суммы в долларах: `Lot Area`, `SalePrice`
  и т.д.);
* плюс `Order` и `PID` — идентификаторы наблюдения.

### 1.3 Ключевые ограничения и гипотезы, явно указанные автором датасета

Официальная документация содержит два раздела, напрямую относящихся к
подготовке данных — **это не наши догадки, а инструкции автора датасета**:

> **Potential Pitfalls (Outliers).** *"There are 5 observations that an
> instructor may wish to remove from the data set... Three of them are
> true outliers (Partial Sales that likely don't represent actual market
> values) and two of them are simply unusual sales... I would recommend
> removing any houses with more than 4000 square feet from the data set."*

То есть: **`Gr Liv Area` (жилая площадь выше уровня земли) > 4000 кв.
футов** — задокументированный критерий выброса, а не эвристика,
придуманная нами постфактум. Ниже (раздел 2.1) увидим, что в этом файле
эти самые 5 наблюдений реально присутствуют и в точности соответствуют
описанию автора.

> **О "типичности" продажи.** *"...if the purpose is to once again create
> a common use model to estimate a typical sale, it is in the modeler's
> best interest to remove any observations that do not seem typical (such
> as foreclosures or family sales)."*

То есть столбец **`Sale Condition`** несёт прямой сигнал о том, отражает
ли цена сделки рыночную стоимость: `Abnorml` — принудительная продажа
(обращение взыскания и т.п.), `Partial` — дом ещё достраивался на
момент оценки, `Family` — продажа между родственниками, `Alloca` —
раздельные сделки по факту единого объекта (например, кондоминиум с
отдельным гаражным юнитом), `AdjLand` — покупка соседнего участка
земли.

> **О гетероскедастичности и целевой переменной.** Документация отдельно
> обсуждает, что дисперсия `SalePrice` растёт вместе с самой ценой
> (типичная картина для цен), и что log-преобразование цены — стандартный
> приём для такого рода данных, хотя и снижает интерпретируемость
> коэффициентов в линейных моделях.

### 1.4 Доменные гипотезы/"уравнения", которые логично проверить

Из практики оценки недвижимости (и здравого смысла) вытекают конкретные
конструкции признаков, которые мы реализуем в разделе 3:

* $\text{TotalSF} = \text{TotalBsmtSF} + \text{1stFlrSF} + \text{2ndFlrSF}$
  — риелторы и оценщики обычно ориентируются на суммарную площадь дома,
  а не на площади по этажам по отдельности;
* $\text{HouseAge} = \text{YrSold} - \text{YearBuilt}$ и
  $\text{RemodAge} = \text{YrSold} - \text{YearRemodAdd}$ — возраст дома
  и возраст последнего ремонта на момент продажи обычно значимее, чем
  сам год постройки;
* порядковые оценки качества (`Ex/Gd/TA/Fa/Po`) естественно кодировать
  числовой шкалой $5/4/3/2/1$ (и $0$ для `NA`), а не как произвольные
  категории без порядка — это соответствует их семантике по документации;
* $\text{TotalBath} = \text{FullBath} + 0{,}5\cdot\text{HalfBath} +
  \text{BsmtFullBath} + 0{,}5\cdot\text{BsmtHalfBath}$ — стандартная
  агрегированная метрика "число ванных комнат" в объявлениях о продаже.

In [ ]:
num_cols = df_raw.select_dtypes(include=np.number).columns
cat_cols = df_raw.select_dtypes(exclude=np.number).columns
print(f"Числовых столбцов: {len(num_cols)}, категориальных: {len(cat_cols)}, всего: {df_raw.shape[1]}")
print(f"Уникальных районов (Neighborhood): {df_raw['Neighborhood'].nunique()}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df_raw["SalePrice"], bins=40, color=style.GREEN, edgecolor="black")
axes[0].set_title("SalePrice (исходная шкала)")
axes[0].set_xlabel("$")

axes[1].hist(np.log1p(df_raw["SalePrice"]), bins=40, color=style.BLUE, edgecolor="black")
axes[1].set_title("log1p(SalePrice)")
axes[1].set_xlabel("log(1 + $)")
plt.tight_layout()
plt.show()

print("Асимметрия (skewness) SalePrice: {:.3f}".format(df_raw["SalePrice"].skew()))
print("Асимметрия (skewness) log1p(SalePrice): {:.3f}".format(np.log1p(df_raw["SalePrice"]).skew()))

Как и предупреждает документация: `SalePrice` заметно скошена вправо
(есть немало дорогих домов, вытягивающих правый хвост) — после
`log1p`-преобразования асимметрия резко снижается и распределение
становится гораздо ближе к нормальному. В ноутбуке 07 будем сравнивать
модели как на исходной, так и на логарифмированной цели.

## 2. Подготовка данных

### 2.1 Выбросы: `Gr Liv Area` > 4000 кв. футов

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df_raw["Gr Liv Area"], df_raw["SalePrice"], color=style.GREEN,
           edgecolor="black", alpha=0.5, s=25)
outlier_mask = df_raw["Gr Liv Area"] > 4000
ax.scatter(df_raw.loc[outlier_mask, "Gr Liv Area"], df_raw.loc[outlier_mask, "SalePrice"],
           color=style.ERROR_RED, edgecolor="black", s=90, zorder=5,
           label="Gr Liv Area > 4000 (выбросы по документации)")
ax.axvline(4000, color=style.ERROR_RED, linestyle="--", alpha=0.6)
ax.set_xlabel("Gr Liv Area (жилая площадь, кв. футы)")
ax.set_ylabel("SalePrice, $")
ax.set_title("Проверка задокументированного критерия выброса")
ax.legend()
plt.tight_layout()
plt.show()

print("Найдено потенциальных выбросов (Gr Liv Area > 4000):", outlier_mask.sum())
display(df_raw.loc[outlier_mask, ["Order", "PID", "Gr Liv Area", "SalePrice", "Sale Condition"]])

Ровно **5** наблюдений, как и написано в документации, — и их
природа в точности соответствует авторскому описанию: три продажи с
`Sale Condition == "Partial"` (дом ещё достраивался на момент оценки) и
непропорционально низкой ценой относительно площади — это и есть "true
outliers", не отражающие рыночную стоимость; и два дома с
действительно большой площадью (4316 и 4476 кв. футов), проданных
дорого, но соразмерно площади — "unusual sales, priced relatively
appropriately", которые De Cock прямо советовал **не путать** с первыми
тремя при принятии решения об удалении. Тем не менее для задачи
построения типовой модели цены его итоговая рекомендация — убрать все
пять по единому порогу площади (`> 4000`), что мы и делаем.

In [ ]:
df_clean = df_raw.loc[~outlier_mask].copy()
print("Размер после фильтра выбросов:", df_clean.shape)

### 2.2 "Типичность" продажи (`Sale Condition`)

In [ ]:
sale_cond_counts = df_clean["Sale Condition"].value_counts()
print(sale_cond_counts)

fig, ax = plt.subplots(figsize=(7, 4))
colors = [style.GREEN if c == "Normal" else style.category_color(i + 1)
          for i, c in enumerate(sale_cond_counts.index)]
ax.bar(sale_cond_counts.index, sale_cond_counts.values, color=colors)
ax.set_ylabel("Число сделок")
ax.set_title("Sale Condition — распределение по выборке")
plt.tight_layout()
plt.show()

print("\nМедианная цена по типу сделки:")
print(df_clean.groupby("Sale Condition")["SalePrice"].median().sort_values())

Как и предупреждала документация, доля нетипичных сделок вполне
ощутима (около 18% всех продаж) — и медианные цены по типам сделки
действительно заметно различаются: `Abnorml` (принудительные продажи)
и `AdjLand` (покупка примыкающей земли — часто формальность) стабильно
дешевле, `Partial` (новостройки) — обычно дороже. Это подтверждает, что
`Sale Condition` несёт реальный ценовой сигнал, а не просто
"техническую" пометку.

**Решение (осознанный, а не автоматический выбор):** вместо того чтобы
молча удалить эти строки (что теряет данные и требует переобучать
модель под другую задачу), создаём бинарный признак `Is_Normal_Sale` —
модель сможет сама научиться взвешивать нетипичные продажи, а в
ноутбуке 07 отдельно сравним качество с этим признаком и без него —
именно то компромиссное решение, о котором пишет сама документация
("зависит от цели модели").

In [ ]:
df_clean["Is_Normal_Sale"] = (df_clean["Sale Condition"] == "Normal").astype(int)
print(df_clean["Is_Normal_Sale"].value_counts())

### 2.3 Пропуски — с опорой на документацию

Ключевая идея: для многих
столбцов **`NA` — это не пропуск, а отдельная содержательная категория**
"у объекта нет этой характеристики". Например:

```
Pool QC (Ordinal): ... NA  No Pool
Fence (Ordinal):   ... NA  No Fence
FireplaceQu:       ... NA  No Fireplace
Garage Type/Finish/Qual/Cond: ... NA  No Garage
Bsmt Qual/Cond/Exposure/BsmtFinType1/2: ... NA  No Basement
Alley:             ... NA  No alley access
Misc Feature:      ... NA  None
Mas Vnr Type:      ... None  None (нет облицовки)
```

Если заполнить такие столбцы медианой/модой или, того хуже, выбросить
строки с пропусками — модель потеряет содержательный сигнал ("у дома
просто нет бассейна/подвала/гаража"), который на самом деле не пропуск,
а факт.

In [ ]:
missing = df_clean.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

Разберём это по группам, опираясь на документацию — и, что важно,
**проверим данные напрямую**, а не просто поверим общей логике:

* **`Pool QC`, `Misc Feature`, `Alley`, `Fence`, `Fireplace Qu`,
  `Mas Vnr Type`** — `NA`/пропуск здесь = "этой характеристики нет".
  Заполняем строкой `"None"` (для `Mas Vnr Area` — числовым `0`).
* **Гараж.** У `Garage Type` пропущено на **2 строки меньше**, чем у
  `Garage Yr Blt`/`Garage Finish`/`Garage Qual`/`Garage Cond`. Прямая
  проверка ниже показывает: это ровно 2 дома, где `Garage Type` *есть*
  (гараж физически существует), но его качество/состояние/год постройки
  почему-то не зафиксированы — **настоящий пропуск**, а не "нет
  гаража"; у одного из этих двух домов пусты ещё и числовые
  `Garage Cars`/`Garage Area`. Заполнять их `"None"`/нулём было бы
  неверно: это записало бы существующий гараж как отсутствующий.
* **Подвал.** Аналогично: `Bsmt Exposure` пропущен на 2 строки больше,
  чем `Bsmt Qual`/`Bsmt Cond`, а `BsmtFin Type 2` — на 1 строку больше.
  Это тоже настоящие точечные пропуски на фоне реально существующего
  подвала (`Bsmt Qual` заполнен, `Total Bsmt SF` > 0), а не "нет
  подвала". Отдельно проверяем и обратный случай: есть ли дом, где
  подвала действительно нет, но при этом заполнены не все связанные
  числовые поля (`BsmtFin SF 1/2`, `Bsmt Unf SF`, `Total Bsmt SF`,
  `Bsmt Full/Half Bath`) — их в этом случае нужно явно занулить, а не
  просто оставить как есть.
* **`Electrical`** — по документации пропусков в принципе быть не
  должно (это не входит в список "NA как категория"); и действительно,
  здесь мы находим **ровно один** пропуск на весь датасет — классический
  случай единичной точечной ошибки ввода данных, которую разумно
  заполнить самой частой категорией (модой).

In [ ]:
garage_cols = ["Garage Type", "Garage Yr Blt", "Garage Finish", "Garage Qual", "Garage Cond"]
garage_exists_but_gap = df_clean["Garage Type"].notna() & df_clean["Garage Qual"].isna()
print("Гараж физически есть, но часть полей не заполнена:", garage_exists_but_gap.sum(), "дом(а)")
display(df_clean.loc[garage_exists_but_gap, ["Order", "PID"] + garage_cols
                      + ["Garage Cars", "Garage Area"]])

bsmt_cols = ["Bsmt Qual", "Bsmt Cond", "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2"]
bsmt_num_cols = ["BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF", "Total Bsmt SF",
                  "Bsmt Full Bath", "Bsmt Half Bath"]
bsmt_exists_exposure_gap = df_clean["Bsmt Qual"].notna() & df_clean["Bsmt Exposure"].isna()
bsmt_exists_fintype2_gap = df_clean["Bsmt Qual"].notna() & df_clean["BsmtFin Type 2"].isna()
print("\nПодвал есть, но не заполнена Bsmt Exposure:", bsmt_exists_exposure_gap.sum(), "дома")
display(df_clean.loc[bsmt_exists_exposure_gap, ["Order", "PID"] + bsmt_cols + ["Total Bsmt SF"]])
print("\nПодвал есть, но не заполнен BsmtFin Type 2:", bsmt_exists_fintype2_gap.sum(), "дом")
display(df_clean.loc[bsmt_exists_fintype2_gap, ["Order", "PID"] + bsmt_cols + ["Total Bsmt SF"]])

bsmt_fully_missing = df_clean[bsmt_cols + bsmt_num_cols].isna().all(axis=1)
print("\nДом(а), где ВСЕ поля подвала (и категориальные, и числовые) пусты",
      "(настоящее «нет подвала», нужно заполнить числовые нулями):",
      bsmt_fully_missing.sum())
display(df_clean.loc[bsmt_fully_missing, ["Order", "PID"] + bsmt_cols + bsmt_num_cols])

print("\nПропусков Electrical:", df_clean["Electrical"].isna().sum())
display(df_clean.loc[df_clean["Electrical"].isna(), ["Order", "PID", "Electrical"]])

Подтвердилось по всем пунктам — и нашёлся ещё один настоящий, но
уже "зеркальный" случай: дом (`Order=1342`), где подвала действительно
нет (все категориальные поля подвала пусты), но при этом пусты и
связанные числовые поля (`BsmtFin SF 1/2`, `Bsmt Unf SF`,
`Total Bsmt SF`, `Bsmt Full/Half Bath`) — их нужно явно заполнить
нулями (площадь несуществующего подвала — 0, а не пропуск), иначе
после заполнения категориальных полей `"None"` в данных всё равно
останутся "дыры" в числовых столбцах.

Заполняем точечные "настоящие" пропуски (при реально существующих
гараже/подвале) модой/медианой среди объектов, где соответствующая
характеристика реально присутствует — а не общей меткой "нет
гаража"/"нет подвала", и отдельно зануляем числовые поля там, где
характеристики действительно нет.

* **`Lot Frontage`** (длина фасада вдоль улицы, в футах) — самый
  массовый настоящий пропуск (у каждого дома есть какой-то фасад,
  просто не всегда зафиксирован). Заполнять глобальным средним/медианой
  не лучший вариант — соседние дома в одном районе (`Neighborhood`)
  обычно стоят на участках очень похожей формы и ширины (районы
  застраивались едиными очередями). Заполняем **медианой по своему
  `Neighborhood`** — доменно обоснованная группировка, а не глобальная
  статистика.

In [ ]:
frontage_by_hood = df_clean.groupby("Neighborhood")["Lot Frontage"].median().sort_values()

fig, ax = plt.subplots(figsize=(11, 4.5))
sns.boxplot(data=df_clean, x="Neighborhood", y="Lot Frontage",
            order=frontage_by_hood.index, ax=ax, color=style.MINT)
ax.set_title("Lot Frontage по районам — обоснование group-wise импутации")
plt.setp(ax.get_xticklabels(), rotation=75, ha="right")
plt.tight_layout()
plt.show()

print("Пропусков Lot Frontage:", df_clean["Lot Frontage"].isna().sum(),
      f"из {len(df_clean)} ({df_clean['Lot Frontage'].isna().mean():.1%})")

Разброс `Lot Frontage` **внутри** большинства районов заметно уже,
чем разброс между районами (например, `NridgHt`/`Veenker` — широкие
участки, `BrDale`/`MeadowV` — узкие) — это наглядно подтверждает, что
районная медиана является намного более точным предположением о
пропущенном значении, чем глобальная медиана по всей выборке.

In [ ]:
def clean_ames(df):
    df = df.copy()

    # --- 1. Выбросы (задокументированный критерий) ---
    df = df.loc[df["Gr Liv Area"] <= 4000].copy()

    # --- 2. "NA = отдельная категория" (по документации) ---
    none_category_cols = [
        "Pool QC", "Misc Feature", "Alley", "Fence", "Fireplace Qu", "Mas Vnr Type",
    ]
    for c in none_category_cols:
        df[c] = df[c].fillna("None")
    df["Mas Vnr Area"] = df["Mas Vnr Area"].fillna(0)

    # --- 3. Гараж: сперва точечный пропуск при реально существующем гараже,
    #     затем массовое "нет гаража" ---
    has_garage = df["Garage Type"].notna()
    garage_gap = has_garage & df["Garage Qual"].isna()
    for c in ["Garage Finish", "Garage Qual", "Garage Cond"]:
        mode_val = df.loc[has_garage, c].mode()[0]
        df.loc[garage_gap, c] = df.loc[garage_gap, c].fillna(mode_val)
    df.loc[garage_gap, "Garage Yr Blt"] = df.loc[garage_gap, "Year Built"]
    for c in ["Garage Cars", "Garage Area"]:
        median_val = df.loc[has_garage, c].median()
        df.loc[garage_gap, c] = df.loc[garage_gap, c].fillna(median_val)

    for c in ["Garage Type", "Garage Finish", "Garage Qual", "Garage Cond"]:
        df[c] = df[c].fillna("None")
    df["Garage Yr Blt"] = df["Garage Yr Blt"].fillna(df["Year Built"])
    for c in ["Garage Cars", "Garage Area"]:
        df[c] = df[c].fillna(0)

    # --- 4. Подвал: точечные пропуски при реально существующем подвале,
    #     затем массовое "нет подвала" (включая числовые поля) ---
    has_bsmt = df["Bsmt Qual"].notna()
    for col in ["Bsmt Exposure", "BsmtFin Type 2"]:
        gap = has_bsmt & df[col].isna()
        if gap.any():
            mode_val = df.loc[has_bsmt, col].mode()[0]
            df.loc[gap, col] = mode_val

    for c in ["Bsmt Qual", "Bsmt Cond", "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2"]:
        df[c] = df[c].fillna("None")
    for c in ["BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF", "Total Bsmt SF",
              "Bsmt Full Bath", "Bsmt Half Bath"]:
        df[c] = df[c].fillna(0)

    # --- 5. Электрика: единичная точечная ошибка ввода -> мода ---
    if df["Electrical"].isna().any():
        df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

    # --- 6. Настоящий массовый пропуск -> group-wise импутация ---
    df["Lot Frontage"] = df.groupby("Neighborhood")["Lot Frontage"].transform(
        lambda s: s.fillna(s.median()))
    df["Lot Frontage"] = df["Lot Frontage"].fillna(df["Lot Frontage"].median())

    return df


df_clean = clean_ames(df_clean)
remaining = df_clean.isna().sum()
remaining = remaining[remaining > 0]
print("Столбцов с оставшимися пропусками:", len(remaining))
print(remaining)
print("\nИтоговый размер после очистки:", df_clean.shape)

## 3. Feature engineering

Строим признаки, явно опирающиеся на доменные соображения раздела 1.4:
суммарную площадь, возраст дома/ремонта/гаража, числовое кодирование
порядковых оценок качества и агрегированное число ванных комнат.

In [ ]:
QUALITY_MAP = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0}
quality_cols = ["Exter Qual", "Exter Cond", "Bsmt Qual", "Bsmt Cond",
                "Heating QC", "Kitchen Qual", "Fireplace Qu",
                "Garage Qual", "Garage Cond", "Pool QC"]


def engineer_features(df):
    df = df.copy()

    # --- площади и возраст ---
    df["TotalSF"] = df["Total Bsmt SF"] + df["1st Flr SF"] + df["2nd Flr SF"]
    df["HouseAge"] = (df["Yr Sold"] - df["Year Built"]).clip(lower=0)
    df["RemodAge"] = (df["Yr Sold"] - df["Year Remod/Add"]).clip(lower=0)
    df["GarageAge"] = (df["Yr Sold"] - df["Garage Yr Blt"]).clip(lower=0)
    df["IsRemodeled"] = (df["Year Remod/Add"] != df["Year Built"]).astype(int)

    # --- ванные комнаты ---
    df["TotalBath"] = (df["Full Bath"] + 0.5 * df["Half Bath"]
                        + df["Bsmt Full Bath"] + 0.5 * df["Bsmt Half Bath"])

    # --- числовое кодирование порядковых оценок качества ---
    for c in quality_cols:
        df[c + " Score"] = df[c].map(QUALITY_MAP).fillna(0).astype(int)

    df["OverallGrade"] = df["Overall Qual"] * df["Overall Cond"]

    # --- бинарные индикаторы наличия ---
    df["HasPool"] = (df["Pool Area"] > 0).astype(int)
    df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)
    df["Has2ndFlr"] = (df["2nd Flr SF"] > 0).astype(int)
    df["HasBsmt"] = (df["Total Bsmt SF"] > 0).astype(int)
    df["HasGarage"] = (df["Garage Area"] > 0).astype(int)

    # --- целевая переменная в лог-шкале (см. раздел 1) ---
    df["LogSalePrice"] = np.log1p(df["SalePrice"])

    return df


df_prepared = engineer_features(df_clean)
new_cols = [c for c in df_prepared.columns if c not in df_clean.columns]
print(f"Добавлено {len(new_cols)} новых признаков:")
print(new_cols)
print("\nИтоговый размер:", df_prepared.shape)

In [ ]:
engineered_numeric = ["TotalSF", "HouseAge", "RemodAge", "GarageAge", "TotalBath",
                      "OverallGrade"] + [c + " Score" for c in quality_cols]
corrs = df_prepared[engineered_numeric + ["SalePrice"]].corr()["SalePrice"].drop("SalePrice")
corrs = corrs.sort_values()

fig, ax = plt.subplots(figsize=(7, 6))
colors = [style.ERROR_RED if v < 0 else style.GREEN for v in corrs.values]
ax.barh(corrs.index, corrs.values, color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Корреляция с SalePrice")
ax.set_title("Насколько сконструированные признаки связаны с ценой")
plt.tight_layout()
plt.show()

На полных данных картина стала ещё более убедительной: `TotalSF`,
`OverallGrade` и оценки качества (`Exter Qual`, `Kitchen Qual`, `Bsmt
Qual`) — сильнейшие положительные предикторы цены, что полностью
согласуется с интуицией оценщика недвижимости: размер дома и общее
качество отделки — главные ценообразующие факторы. Возрастные признаки
(`HouseAge`, `GarageAge`, `RemodAge`) закономерно отрицательно
коррелируют с ценой.

## Сохранение результата для ноутбука 07

In [ ]:
OUT_PATH = join('data', "ames_prepared.csv")
df_prepared.to_csv(OUT_PATH, index=False)
print("Сохранено:", OUT_PATH, "| размер:", df_prepared.shape)

## Итоги ноутбука

* **Доменная информация** получена из официальной документации автора
  датасета — описания всех колонок, задокументированный критерий
  выброса (`Gr Liv Area > 4000`), явное указание на связь
  `Sale Condition` с "типичностью" сделки, обсуждение
  гетероскедастичности цены.
* **Очистка** выполнена по документации и проверена на реальных
  данных: все 5 задокументированных выбросов найдены и в точности
  соответствуют описанию автора; отдельно найдены и корректно
  обработаны точечные "настоящие" пропуски на фоне реально
  существующих гаража/подвала (2, 2 и 1 наблюдение соответственно) и
  единственный пропуск в `Electrical` — в отличие от массовых
  "NA как категория"; `Lot Frontage` импутирован медианой **по
  району**, с наглядным обоснованием на boxplot.
* **Feature engineering** дал признаки, обоснованные логикой оценки
  недвижимости: суммарная площадь, возраст дома/ремонта/гаража,
  числовые шкалы качества, агрегированное число ванных, логарифм цены.

Результат — `ames_prepared.csv` — используется в ноутбуке
**`07_modeling_cross_validation.ipynb`** для моделирования и честной
кросс-валидации.

In [ ]:
print(f"Итоговый датасет: {df_prepared.shape[0]} строк, {df_prepared.shape[1]} столбцов")